In [1]:
import pandas as pd
import numpy as np
import time
import glob
import duckdb
import statsbombpy as sb

In [2]:
# create a connection to an in-memory DuckDB database
con = duckdb.connect("../data/database/futbol-analysis.db")


In [3]:
# create competitions table from json file
con.execute("""
CREATE OR REPLACE TABLE competitions AS
    SELECT * FROM '../data/open-data-master/data/competitions.json';
""")


In [4]:

# create matches table from json files
con.execute("""
CREATE OR REPLACE TABLE matches AS
    SELECT * FROM '../data/open-data-master/data/matches/*/*.json';
""")


In [5]:

# create lineups table from json files
con.execute("""
CREATE OR REPLACE TABLE lineups AS
    SELECT * FROM '../data/open-data-master/data/lineups/*.json';
""")


In [6]:
# create events table from json files
con.execute("""
CREATE OR REPLACE TABLE events AS
    SELECT * FROM read_json(
        '../data/open-data-master/data/events/*.json',
        format='auto',
        union_by_name=true
    );
""")

In [7]:
# check the tables created and get their schema

print(con.execute("""
SHOW TABLES;
""").df())

competitions_table_info = con.execute("""
PRAGMA table_info('competitions');
""").df()

matches_table_info = con.execute("""
PRAGMA table_info('matches');
""").df()

lineups_table_info = con.execute("""
PRAGMA table_info('lineups');
""").df()

events_table_info = con.execute("""
PRAGMA table_info('events');
""").df()

           name
0  competitions
1        events
2       lineups
3       matches


In [8]:
competitions_table_info

,cid,name,type,notnull,dflt_value,pk
0,0,competition_id,BIGINT,False,None,False
1,1,season_id,BIGINT,False,None,False
2,2,country_name,VARCHAR,False,None,False
3,3,competition_name,VARCHAR,False,None,False
4,4,competition_gender,VARCHAR,False,None,False
5,5,competition_youth,BOOLEAN,False,None,False
6,6,competition_international,BOOLEAN,False,None,False
7,7,season_name,VARCHAR,False,None,False
8,8,match_updated,VARCHAR,False,None,False
9,9,match_updated_360,VARCHAR,False,None,False


In [9]:
# set competition_id and season_id to primary key in competitions table
con.execute("""
ALTER TABLE competitions
ADD PRIMARY KEY (competition_id, season_id);
""")

In [10]:
matches_table_info

,cid,name,type,notnull,dflt_value,pk
0,0,match_id,BIGINT,False,None,False
1,1,match_date,DATE,False,None,False
2,2,kick_off,TIME,False,None,False
3,3,competition,"STRUCT(competition_id BIGINT, country_name VAR...",False,None,False
4,4,season,"STRUCT(season_id BIGINT, season_name VARCHAR)",False,None,False
5,5,home_team,"STRUCT(home_team_id BIGINT, home_team_name VAR...",False,None,False
6,6,away_team,"STRUCT(away_team_id BIGINT, away_team_name VAR...",False,None,False
7,7,home_score,BIGINT,False,None,False
8,8,away_score,BIGINT,False,None,False
9,9,match_status,VARCHAR,False,None,False


In [11]:
# set match_id to primary key in matches table
con.execute("""
ALTER TABLE matches
ADD PRIMARY KEY (match_id);
""")

In [12]:
lineups_table_info

,cid,name,type,notnull,dflt_value,pk
0,0,team_id,BIGINT,False,None,False
1,1,team_name,VARCHAR,False,None,False
2,2,lineup,"STRUCT(player_id BIGINT, player_name VARCHAR, ...",False,None,False


In [13]:
# examine lineup structure from lineups table
lineup_structure = con.execute("""
SELECT lineups
FROM lineups
LIMIT 1;
""").df().iloc[0,0]
lineup_structure

{'team_id': 217,
 'team_name': 'Barcelona',
 'lineup': [{'player_id': 3109,
   'player_name': 'Malcom Filipe Silva de Oliveira',
   'player_nickname': 'Malcom',
   'jersey_number': 14,
   'country': {'id': 31, 'name': 'Brazil'},
   'cards': [],
   'positions': []},
  {'player_id': 3501,
   'player_name': 'Philippe Coutinho Correia',
   'player_nickname': 'Philippe Coutinho',
   'jersey_number': 7,
   'country': {'id': 31, 'name': 'Brazil'},
   'cards': [],
   'positions': [{'position_id': 15,
     'position': 'Left Center Midfield',
     'from': '45:00',
     'to': '45:00',
     'from_period': 2,
     'to_period': 2,
     'start_reason': 'Tactical Shift',
     'end_reason': 'Substitution - On (Tactical)'},
    {'position_id': 2,
     'position': 'Right Back',
     'from': '45:00',
     'to': '76:14',
     'from_period': 2,
     'to_period': 2,
     'start_reason': 'Substitution - On (Tactical)',
     'end_reason': 'Tactical Shift'},
    {'position_id': 21,
     'position': 'Left Wing',

In [14]:
events_table_info

,cid,name,type,notnull,dflt_value,pk
0,0,id,UUID,False,None,False
1,1,index,BIGINT,False,None,False
2,2,period,BIGINT,False,None,False
3,3,timestamp,TIME,False,None,False
4,4,minute,BIGINT,False,None,False
5,5,second,BIGINT,False,None,False
6,6,type,"STRUCT(id BIGINT, ""name"" VARCHAR)",False,None,False
7,7,possession,BIGINT,False,None,False
8,8,possession_team,"STRUCT(id BIGINT, ""name"" VARCHAR)",False,None,False
9,9,play_pattern,"STRUCT(id BIGINT, ""name"" VARCHAR)",False,None,False


In [15]:
# set id to primary key in events table
con.execute("""
ALTER TABLE events
ADD PRIMARY KEY (id);
""")